<a href="https://colab.research.google.com/github/Moh-lab-sys/Psychology-Diagnosis/blob/main/PsychologyDiagnosis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing the libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from imblearn.over_sampling import SMOTEN

## Importing the dataset

In [ ]:
dataset = pd.read_csv('falsified_dataset.csv', sep=None, engine='python')
dataset = dataset.dropna(subset=[dataset.columns[1], dataset.columns[2]])

X = dataset.iloc[:, 1].values
y = dataset.iloc[:, 2].values


In [ ]:
print(y)

['Bipolar' 'Bipolar' 'Bipolar' ... 'Bipolar' 'Bipolar' 'Bipolar']


## Balance the dataset

In [ ]:
sm = SMOTEN(random_state=42)
X, y = sm.fit_resample(X.reshape(-1,1),y)
print(np.unique(y, return_counts=True))

(array(['Anxiety', 'Bipolar', 'Depression', 'Stress',
       'personality disorder'], dtype=object), array([20184, 20184, 20184, 20184, 20184]))


## split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0, stratify=y)
X_validation, X_test, y_validation, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 0, stratify=y_test)

In [ ]:
X_train = [str(x).strip() for x in X_train if str(x).strip() not in ('nan', 'None', 'none', '')]
X_test = [str(x).strip() for x in X_test if str(x).strip() not in ('nan', 'None', 'none', '')]


## sentence transformer encoding for independent variable

In [ ]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
X_train = model.encode(X_train)
print(X_train)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 0.06012905  0.1515782   0.26135555 ...  0.15504548 -0.4805258
  -0.01523671]
 [ 0.01955861 -0.163457    0.08812991 ...  0.47328717 -0.44640803
   0.19180518]
 [ 0.01955861 -0.163457    0.08812991 ...  0.47328717 -0.44640803
   0.19180518]
 ...
 [-0.06209433 -0.070294    0.22591108 ...  0.00782677  0.05004152
  -0.1662988 ]
 [ 0.33442768  0.37684447 -0.20383039 ...  0.24377646 -0.00131954
   0.19855715]
 [-0.09619263 -0.08977277  0.1100855  ...  0.02336882 -0.16604844
  -0.02867691]]


In [ ]:
print(X_train)

[[ 0.06012905  0.1515782   0.26135555 ...  0.15504548 -0.4805258
  -0.01523671]
 [ 0.01955861 -0.163457    0.08812991 ...  0.47328717 -0.44640803
   0.19180518]
 [ 0.01955861 -0.163457    0.08812991 ...  0.47328717 -0.44640803
   0.19180518]
 ...
 [-0.06209433 -0.070294    0.22591108 ...  0.00782677  0.05004152
  -0.1662988 ]
 [ 0.33442768  0.37684447 -0.20383039 ...  0.24377646 -0.00131954
   0.19855715]
 [-0.09619263 -0.08977277  0.1100855  ...  0.02336882 -0.16604844
  -0.02867691]]


## Label encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
l = LabelEncoder()
y_train = l.fit_transform(y_train)
y_test = l.transform(y_test)
print(l.classes_)

['Anxiety' 'Bipolar' 'Depression' 'Stress' 'personality disorder']


In [ ]:

print(y_test)
print(y_train)

[0 0 2 ... 2 1 1]
[4 0 0 ... 3 1 4]


## validation on SVM

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
X_validation = [str(x).strip() for x in X_validation if str(x).strip() not in ('nan', 'None', 'none', '')]

X_validation = model.encode(X_validation)
parameters = {'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01],
    'kernel': ['rbf', 'linear'],
    'class_weight': ['balanced', None]}
grid = GridSearchCV(param_grid=parameters, estimator=SVC(), scoring='accuracy', cv=5)
grid.fit(X_validation, y_validation)
print(grid.best_params_)

## Training the logistic regression model on the Training set

In [ ]:

from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 0)
classifier.fit(X_train, y_train)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(random_state=0)

## Predicting the Test set results

In [ ]:
X_test = model.encode(X_test)

In [ ]:

y_pred = classifier.predict(X_test)
y_pred_class = l.inverse_transform(y_pred)

KeyboardInterrupt: 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RF = RandomForestClassifier()
RF.fit(X_train, y_train)
y_pred2 = RF.predict(X_test)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
KNN = KNeighborsClassifier()
KNN.fit(X_train, y_train)
y_pred3 = KNN.predict(X_test)

In [ ]:
from sklearn.naive_bayes import GaussianNB
NB = GaussianNB()
NB.fit(X_train, y_train)
y_pred4 = NB.predict(X_test)

In [ ]:
from sklearn.svm import SVC
SVM = SVC()
SVM.fit(X_train, y_train)
y_pred5 = SVM.predict(X_test)

In [ ]:
from xgboost import XGBClassifier
XGB = XGBClassifier()
XGB.fit(X_train, y_train)
y_pred6 = XGB.predict(X_test)

## Mental health detection

In [ ]:
my_try = 'I feel like I am unable to control my feelings'
my_try = model.encode(my_try)

predict = SVM.predict([my_try])
predict2 = l.inverse_transform(predict)
print(predict2)


['personality disorder']


## Accuracy rates & Confusion Matrix

In [ ]:


from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(accuracy_score(y_test, y_pred))

cm2 = confusion_matrix(y_test, y_pred2)
print(cm2)
print(accuracy_score(y_test, y_pred2))

cm3 = confusion_matrix(y_test, y_pred3)
print(cm3)
print(accuracy_score(y_test, y_pred3))

cm4 = confusion_matrix(y_test, y_pred4)
print(cm4)
print(accuracy_score(y_test, y_pred4))

cm5 = confusion_matrix(y_test, y_pred5)
print(cm5)
print(accuracy_score(y_test, y_pred5))

cm6 = confusion_matrix(y_test, y_pred6)
print(cm6)
print(accuracy_score(y_test, y_pred6))
#


[[2440   26   21    9   27]
 [   8 2280   85   61   89]
 [   6  133 1880   14  490]
 [   6   83   40 2361   33]
 [   6   86  450   19 1962]]
0.8658739595719381
[[2408   43   16    0   56]
 [   0 2350   66    2  105]
 [   0  253 1736    2  532]
 [   0  106   68 2269   80]
 [   0  181  383    0 1959]]
0.8499405469678953
[[2422   13   38    6   44]
 [  29 1961  260   67  206]
 [   9   69 1907   20  518]
 [  10   63   62 2322   66]
 [  11   70  607   22 1813]]
0.8263971462544589
[[2372   42   37    0   72]
 [   0 2137  239    0  147]
 [   0  307 1814    0  402]
 [   0   74  125 2202  122]
 [   0  314  892    0 1317]]
0.7801823226317876
[[2434   30   19    7   33]
 [   2 2383   45   36   57]
 [   0  120 1936   12  455]
 [   3   72   27 2387   34]
 [   3   63  412   14 2031]]
0.885533095521205
[[2443   25   17    2   36]
 [   2 2337   80   29   75]
 [   1  147 1884   10  481]
 [   1   74   51 2358   39]
 [   4   83  472    7 1957]]
0.8703131193024177


## BoxPlot

In [ ]:
plt.boxplot(X_train)
plt.title('BoxPlot')
plt.ylabel('X train')
plt.show()

## ROC curve

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_estimator(model, X_test, y_test)

plt.plot([0, 1], [0, 1], color="gray", linestyle="--")

plt.title("ROC Curve: Normal vs. Attack")
plt.show()